# CareLoom: Database-Connected Ops Agent — Larkspur Electronics

Larkspur's ops team doesn't want to write SQL every time someone asks "how many orders
are still processing" or "what's Asha Rao's order history." This notebook builds the
CareLoom agent that answers those questions directly against Larkspur's own operational
database — a local, serverless SQLite file with customers, orders, order items, and
support tickets.

What it covers:
- a local serverless SQLite database
- agent tools for schema inspection and read-only querying
- Groq for the agent LLM
- structured output from the agent into database rows
- a simple model split for different agent tasks

LangChain's SQL-agent guide shows the core workflow: inspect tables and schemas, decide what is relevant, generate a query, execute it, and handle SQL mistakes carefully. The tools docs show the basic `@tool` pattern, and the structured-output docs show that `create_agent` can return validated data in `structured_response`.

## Learning goals

By the end of this notebook, you should be able to:

1. Work with a local SQLite database only.
2. Build tools for table listing, schema inspection, and read-only querying.
3. Use Groq as the agent model.
4. Return structured output from the agent.
5. Save the structured result back into SQLite.
6. Use different models for SQL reasoning and row extraction.

## 1) Install packages


In [ ]:
%pip install -qU \
    python-dotenv requests \
    langchain langchain-core langchain-groq \
    sqlalchemy pydantic


## 2) Environment variables

Create a `.env` file with the values you actually have:

```env
GROQ_API_KEY=your_groq_api_key
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=careloom-db-agent-sqlite
```

LangChain's structured-output and agents docs show that `create_agent` can use `response_format` to return typed structured data.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')
LANGSMITH_TRACING = os.getenv('LANGSMITH_TRACING', 'true').lower() == 'true'
LANGSMITH_PROJECT = os.getenv('LANGSMITH_PROJECT', 'careloom-db-agent-sqlite')

print('GROQ_API_KEY set:', bool(GROQ_API_KEY))
print('LANGSMITH_API_KEY set:', bool(LANGSMITH_API_KEY))
print('LANGSMITH_TRACING:', LANGSMITH_TRACING)
print('LANGSMITH_PROJECT:', LANGSMITH_PROJECT)


## 3) Create a local SQLite database

To keep this fully serverless and local, we use a single SQLite file on disk.

The database is Larkspur Electronics' own small operational slice: customers, orders, order items, and support tickets.


In [ ]:
import sqlite3
from pathlib import Path
from datetime import datetime, timezone

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

APP_DB = DATA_DIR / 'app_local.db'

if APP_DB.exists():
    APP_DB.unlink()

conn = sqlite3.connect(APP_DB)
cur = conn.cursor()

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL,
    city TEXT NOT NULL,
    signup_date TEXT NOT NULL
)
""")

cur.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT NOT NULL,
    total_amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cur.execute("""
CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER NOT NULL,
    product_name TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
)
""")

cur.execute("""
CREATE TABLE support_tickets (
    ticket_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    subject TEXT NOT NULL,
    status TEXT NOT NULL,
    created_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

customers = [
    ('Asha Rao', 'Chennai', '2026-01-02'),
    ('Imran Khan', 'Bengaluru', '2026-01-10'),
    ('Neha Patel', 'Mumbai', '2026-01-18'),
    ('Vikram Suresh', 'Delhi', '2026-01-25'),
    ('Divya Menon', 'Pune', '2026-02-01'),
]
cur.executemany(
    'INSERT INTO customers (full_name, city, signup_date) VALUES (?, ?, ?)',
    customers
)

orders = [
    (1, '2026-02-01', 'completed', 2499.0),
    (1, '2026-02-12', 'completed', 899.0),
    (2, '2026-02-14', 'processing', 1499.0),
    (3, '2026-02-20', 'completed', 3199.0),
    (4, '2026-02-22', 'completed', 1899.0),
    (4, '2026-02-25', 'cancelled', 2499.0),
    (5, '2026-03-01', 'processing', 799.0),
]
cur.executemany(
    'INSERT INTO orders (customer_id, order_date, status, total_amount) VALUES (?, ?, ?, ?)',
    orders
)

items = [
    (1, 'Keyboard', 1, 2499.0),
    (2, 'Mouse', 1, 899.0),
    (3, 'Headphones', 1, 1499.0),
    (4, 'Monitor', 1, 3199.0),
    (5, 'Webcam', 1, 1899.0),
    (6, 'Keyboard', 1, 2499.0),
    (7, 'Desk Lamp', 1, 799.0),
]
cur.executemany(
    'INSERT INTO order_items (order_id, product_name, quantity, unit_price) VALUES (?, ?, ?, ?)',
    items
)

tickets = [
    (1, 'Refund request', 'open', '2026-02-03T09:00:00'),
    (2, 'Order status question', 'closed', '2026-02-15T10:30:00'),
    (3, 'Invoice needed', 'open', '2026-02-21T08:45:00'),
    (4, 'Cancelled order refund', 'open', '2026-02-26T11:00:00'),
    (5, 'Warranty question about desk lamp', 'closed', '2026-03-02T09:15:00'),
]
cur.executemany(
    'INSERT INTO support_tickets (customer_id, subject, status, created_at) VALUES (?, ?, ?, ?)',
    tickets
)

conn.commit()
conn.close()

print('Created local SQLite database:', APP_DB.resolve())


## 4) Inspect the database


In [ ]:
conn = sqlite3.connect(APP_DB)
cur = conn.cursor()

cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [row[0] for row in cur.fetchall()]

print('Tables:', tables)

for table in tables:
    cur.execute(f'PRAGMA table_info({table})')
    cols = cur.fetchall()
    print(f'\nSchema for {table}')
    for c in cols:
        print(f'- {c[1]} ({c[2]})')

conn.close()


## 5) Define SQLite tools

The tools docs show the basic `@tool` pattern, and LangChain agents can call those tools during the reasoning loop.


In [ ]:
from langchain.tools import tool

@tool
def sqlite_list_tables() -> str:
    """List tables in the local SQLite database."""
    conn = sqlite3.connect(str(APP_DB))
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    rows = [r[0] for r in cur.fetchall() if not r[0].startswith('sqlite_')]
    conn.close()
    return ', '.join(rows)

@tool
def sqlite_schema(table_name: str) -> str:
    """Return the schema for a SQLite table."""
    conn = sqlite3.connect(str(APP_DB))
    cur = conn.cursor()
    cur.execute(f'PRAGMA table_info({table_name})')
    cols = cur.fetchall()
    conn.close()
    if not cols:
        return f'No such table: {table_name}'
    return '\n'.join([f'{c[1]} ({c[2]})' for c in cols])

@tool
def sqlite_query(query: str) -> str:
    """Run a read-only SELECT/WITH query against SQLite."""
    q = query.strip().lower()
    forbidden = ['insert ', 'update ', 'delete ', 'drop ', 'alter ', 'create ']
    if not q.startswith('select') and not q.startswith('with'):
        return 'Only read-only SELECT/WITH queries are allowed.'
    if any(word in q for word in forbidden):
        return 'Potentially unsafe SQL was blocked.'

    conn = sqlite3.connect(str(APP_DB))
    cur = conn.cursor()
    try:
        cur.execute(query)
        rows = cur.fetchall()
        return str(rows)
    except Exception as e:
        return f'SQLite error: {e}'
    finally:
        conn.close()

print(sqlite_list_tables.invoke({}))


## 6) Choose models per task

A practical split is:
- a stronger model for SQL reasoning
- a smaller, cheaper model for extraction and formatting

This notebook keeps the split simple so you can change it later without rewriting the workflow.


In [ ]:
from langchain_groq import ChatGroq

MODEL_REGISTRY = {
    'sql': os.getenv('GROQ_SQL_MODEL', 'openai/gpt-oss-120b'),
    'extract': os.getenv('GROQ_EXTRACT_MODEL', 'openai/gpt-oss-20b'),
}

def build_model(node_name: str):
    return ChatGroq(
        model=MODEL_REGISTRY[node_name],
        temperature=0,
    )

sql_model = build_model('sql')
extract_model = build_model('extract')

print(MODEL_REGISTRY)


## 7) Define structured output schemas

The structured-output docs say agents can return a validated `structured_response`. A Pydantic model is a convenient schema for that.


In [ ]:
from pydantic import BaseModel, Field

class QueryResult(BaseModel):
    question: str = Field(..., description='Original user question')
    database_name: str = Field(..., description='sqlite')
    sql_query: str = Field(..., description='The SQL query that was used')
    answer: str = Field(..., description='Short answer to the user')
    row_count: int = Field(..., description='How many rows were returned')

class AgentInsight(BaseModel):
    question: str = Field(..., description='Original user question')
    database_name: str = Field(..., description='sqlite')
    summary: str = Field(..., description='Short operational summary')
    sql_query: str = Field(..., description='The query that was executed')
    row_count: int = Field(..., description='Row count from the query')


## 8) Build the SQLite agent

The SQL-agent tutorial describes a loop where the agent inspects tables, checks schemas, builds a query, and executes it with care. Here we keep that pattern but only for SQLite.


In [ ]:
from langchain.agents import create_agent

sql_agent = create_agent(
    sql_model,
    tools=[sqlite_list_tables, sqlite_schema, sqlite_query],
    system_prompt=(
        'You are CareLoom, a SQLite database-connected ops agent for Larkspur Electronics. '
        'Use the SQLite tools to inspect the schema and answer questions. '
        'Always prefer a read-only query. '
        'Keep SQL small and focused.'
    ),
    response_format=QueryResult,
)

print('SQLite agent ready.')


## 9) A safety net for environments without a live Groq connection

Every call from here on asks `sql_agent` or `extract_agent` to actually reach Groq. If that
fails — no key, no network, whatever — we don't want the whole notebook to stop. The
fallback below runs the same SQL directly against the local SQLite file, so the *answer*
stays numerically correct even when the *LLM* isn't available to choose the query or word
the response. What's actually being skipped offline is the model's reasoning, not the data.


In [ ]:
def offline_fallback_query_result(question: str, sql_query: str) -> QueryResult:
    conn = sqlite3.connect(str(APP_DB))
    cur = conn.cursor()
    cur.execute(sql_query)
    rows = cur.fetchall()
    conn.close()
    return QueryResult(
        question=question,
        database_name="sqlite",
        sql_query=sql_query,
        answer=f"[offline-fallback, computed directly against SQLite] {rows}",
        row_count=len(rows),
    )


def ask_sql_agent(agent, question: str, fallback_sql: str):
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]})
        if result.get("structured_response") is None:
            raise ValueError("Agent returned no structured_response.")
        return result
    except Exception as e:
        print(f"[offline-fallback] SQL agent call failed ({type(e).__name__}: {e}).")
        return {"structured_response": offline_fallback_query_result(question, fallback_sql)}


## 10) Ask a SQLite question


In [ ]:
question_1 = 'How many orders are currently marked as completed?'

result_1 = ask_sql_agent(
    sql_agent,
    question_1,
    fallback_sql="SELECT COUNT(*) FROM orders WHERE status = 'completed'",
)

result_1


## 11) Inspect the structured response


In [ ]:
structured_1 = result_1['structured_response']
structured_1


## 12) Save the structured response back into SQLite

This is the structured-output-to-database pattern.

We insert the model-generated row into the local `agent_insights` table.


In [ ]:
# Create audit database

AUDIT_DB = DATA_DIR / "agent_audit.db"

audit_conn = sqlite3.connect(AUDIT_DB)
audit_cur = audit_conn.cursor()

audit_cur.execute("""
CREATE TABLE IF NOT EXISTS agent_insights (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT NOT NULL,
    database_name TEXT NOT NULL,
    sql_query TEXT,
    answer TEXT NOT NULL,
    row_count INTEGER DEFAULT 0,
    created_at TEXT NOT NULL
)
""")

audit_conn.commit()
audit_conn.close()

print("Audit DB ready:", AUDIT_DB)


In [ ]:
audit_conn = sqlite3.connect(str(AUDIT_DB))
audit_cur = audit_conn.cursor()

audit_cur.execute(
    'INSERT INTO agent_insights (question, database_name, sql_query, answer, row_count, created_at) VALUES (?, ?, ?, ?, ?, ?)',
    (
        structured_1.question,
        structured_1.database_name,
        structured_1.sql_query,
        structured_1.answer,
        structured_1.row_count,
        datetime.now(timezone.utc).isoformat(),
    ),
)

audit_conn.commit()
audit_cur.execute('SELECT question, database_name, answer, row_count FROM agent_insights')
saved_rows = audit_cur.fetchall()
audit_conn.close()

saved_rows


## 13) Build a smaller extraction agent

The smaller model is useful for cleanup and row formatting rather than SQL reasoning.


In [ ]:
extract_agent = create_agent(
    extract_model,
    tools=[],
    response_format=AgentInsight,
)

print('Extraction agent ready.')


## 14) Run the extraction agent

Same offline-safety idea as before: if the live call fails, fall back to building the `AgentInsight` directly from the structured SQL result we already have, instead of crashing.


In [ ]:
def offline_fallback_insight(structured: QueryResult) -> AgentInsight:
    return AgentInsight(
        question=structured.question,
        database_name=structured.database_name,
        summary=f"[offline-fallback] {structured.answer}",
        sql_query=structured.sql_query,
        row_count=structured.row_count,
    )


try:
    insight_result = extract_agent.invoke({
        'messages': [{
            'role': 'user',
            'content': (
                'Convert this SQLite agent result into a clean row: '
                f'question={structured_1.question}; '
                f'database={structured_1.database_name}; '
                f'sql={structured_1.sql_query}; '
                f'answer={structured_1.answer}; '
                f'row_count={structured_1.row_count}'
            )
        }]
    })
    if insight_result.get('structured_response') is None:
        raise ValueError('Agent returned no structured_response.')
except Exception as e:
    print(f"[offline-fallback] Extraction agent call failed ({type(e).__name__}: {e}).")
    insight_result = {'structured_response': offline_fallback_insight(structured_1)}

insight_result['structured_response']


## 15) Model choice by node

Use a stronger model for SQL planning and a smaller model for extraction. That keeps the notebook practical and inexpensive while still showing the pattern clearly.


In [ ]:
def choose_node_model(node_name: str) -> str:
    return MODEL_REGISTRY[node_name]

print('sql node model:', choose_node_model('sql'))
print('extract node model:', choose_node_model('extract'))


## 16) A simple SQLite-only router

You can keep routing basic and entirely local. There's only one database here, so this is a placeholder for the day Larkspur adds a second data source — the guided lab notebook later in this module actually needs one.


In [ ]:
def choose_database(question: str) -> str:
    return 'sqlite'

for q in [
    'How many orders are completed?',
    'What is the refund policy?',
]:
    print(q, '->', choose_database(q))


## 17) Why SQLite-only keeps this practical

This version stays simple and easy to run anywhere because it:
- uses only SQLite
- runs locally as a single serverless file
- avoids any external database server configuration
- still demonstrates tools, structured output, and database writes

## Key takeaways

- A small set of `@tool`-wrapped functions (list tables, get schema, run a read-only query) is enough for a real database-connected agent.
- `response_format` on `create_agent` turns free-text answers into validated rows you can store.
- Splitting SQL reasoning and extraction across two models keeps cost down without losing quality where it matters.
- An offline fallback that runs the *same* SQL directly is a good pattern whenever the fallback needs to stay numerically honest, not just non-crashing.

### What's verified where

| Piece | Status |
|---|---|
| SQLite schema, tools, audit-table writes | Executed and verified in this sandbox — all of it is local, so it behaves the same everywhere. |
| `sql_agent` / `extract_agent` live calls | **Not available in this sandbox** (no outbound network to Groq here) — both calls took the `[offline-fallback]` path, which itself ran real SQL against the local database so the numbers you see are correct even without a live model. Confirm the LLM-authored SQL and phrasing on your own machine with a real `GROQ_API_KEY`. |

## References

- SQL agent: https://docs.langchain.com/oss/python/langchain/sql-agent
- Structured output: https://docs.langchain.com/oss/python/langchain/structured-output
- Tools: https://docs.langchain.com/oss/python/langchain/tools
